<a href="https://colab.research.google.com/github/mamassboy/Klasifikasi-Literasi-Keuangan/blob/main/PEMODELAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IMPORT LIBRARY**

In [2]:
# ============================================
# CELL 1: Install & Import — Notebook 03
# Pemodelan ML: Baseline + XGBoost + CatBoost
# ============================================

!pip install xgboost catboost shap imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, re, warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.metrics         import (accuracy_score, f1_score,
                                     classification_report,
                                     confusion_matrix, ConfusionMatrixDisplay)
from xgboost  import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
import shap

# Google Drive
from google.colab import drive

# Unmount dulu jika sebelumnya terpasang
import subprocess
subprocess.run(['fusermount', '-uz', '/content/drive'],
               capture_output=True)

# Mount ulang
drive.mount('/content/drive', force_remount=True)

print("✅ Google Drive berhasil di-mount!")

Mounted at /content/drive
✅ Google Drive berhasil di-mount!


In [3]:
# ============================================
# CELL 2: Load Dataset ML dari Drive
# ============================================

PATH = '/content/drive/MyDrive/literasi-keuangan-ml/data/processed/data_ml.csv'

df_ml = pd.read_csv(PATH)

print(f"✅ Dataset berhasil dimuat!")
print(f"   Baris  : {df_ml.shape[0]}")
print(f"   Kolom  : {df_ml.shape[1]}")
print(f"\n📊 Distribusi Label:")
dist = df_ml['label_literasi'].value_counts()
for k, v in dist.items():
    pct = round(v/len(df_ml)*100, 1)
    bar = '█' * int(pct/2)
    print(f"   {k:<8}: {v:>3} ({pct}%) {bar}")

✅ Dataset berhasil dimuat!
   Baris  : 249
   Kolom  : 32

📊 Distribusi Label:
   Sedang  : 131 (52.6%) ██████████████████████████
   Rendah  :  90 (36.1%) ██████████████████
   Tinggi  :  28 (11.2%) █████


In [4]:
# ============================================
# CELL 3: Preprocessing untuk ML
# Encoding + Split + SMOTE
# ============================================

# ── STEP 1: Pisahkan Fitur & Label ──────────
X = df_ml.drop(columns=['label_literasi'])
y = df_ml['label_literasi']

print(f"✅ STEP 1: Fitur & Label dipisah")
print(f"   X shape : {X.shape}")
print(f"   y shape : {y.shape}")

# ── STEP 2: Encoding Variabel Kategori ──────
# Label Encoding untuk semua kolom kategori
# Kenapa LabelEncoder: XGBoost & CatBoost
# bekerja optimal dengan nilai numerik

le_dict = {}  # simpan encoder untuk inverse nanti
X_encoded = X.copy()

for col in X_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(
        X_encoded[col].astype(str))
    le_dict[col] = le

print(f"\n✅ STEP 2: Label Encoding selesai")
print(f"   Kolom di-encode: {len(le_dict)} kolom")

# ── STEP 3: Encode Label Target ─────────────
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f"\n✅ STEP 3: Target di-encode")
print(f"   Mapping kelas:")
for i, kelas in enumerate(le_target.classes_):
    print(f"   {i} → {kelas}")

# ── STEP 4: Train-Test Split ─────────────────
# 80% train, 20% test
# stratify=y → proporsi kelas tetap terjaga
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"\n✅ STEP 4: Data dibagi Train-Test")
print(f"   Train : {X_train.shape[0]} sampel")
print(f"   Test  : {X_test.shape[0]} sampel")

# ── STEP 5: Handle Imbalanced Data (SMOTE) ──
# SMOTE = Synthetic Minority Over-sampling Technique
# Kenapa: jika kelas tidak seimbang, model cenderung
# bias ke kelas mayoritas
# SMOTE hanya diterapkan ke data TRAIN, bukan test!

dist_train = pd.Series(y_train).value_counts()
rasio = dist_train.max() / dist_train.min()
print(f"\n✅ STEP 5: Cek Imbalance")
print(f"   Rasio : {rasio:.2f}x")

if rasio > 1.5:
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print(f"   ⚠️  SMOTE diterapkan!")
    print(f"   Sebelum: {dict(pd.Series(y_train).value_counts())}")
    print(f"   Sesudah: {dict(pd.Series(y_train_res).value_counts())}")
else:
    X_train_res, y_train_res = X_train, y_train
    print(f"   ✅ Data cukup seimbang, SMOTE tidak diperlukan")

# ── STEP 6: Scaling (untuk Logistic Regression) ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print(f"\n✅ STEP 6: StandardScaler diterapkan")
print(f"\n🚀 Data siap untuk pemodelan!")
print(f"   X_train final : {X_train_res.shape}")
print(f"   X_test        : {X_test.shape}")

✅ STEP 1: Fitur & Label dipisah
   X shape : (249, 31)
   y shape : (249,)

✅ STEP 2: Label Encoding selesai
   Kolom di-encode: 31 kolom

✅ STEP 3: Target di-encode
   Mapping kelas:
   0 → Rendah
   1 → Sedang
   2 → Tinggi

✅ STEP 4: Data dibagi Train-Test
   Train : 199 sampel
   Test  : 50 sampel

✅ STEP 5: Cek Imbalance
   Rasio : 4.77x
   ⚠️  SMOTE diterapkan!
   Sebelum: {1: np.int64(105), 0: np.int64(72), 2: np.int64(22)}
   Sesudah: {0: np.int64(105), 1: np.int64(105), 2: np.int64(105)}

✅ STEP 6: StandardScaler diterapkan

🚀 Data siap untuk pemodelan!
   X_train final : (315, 31)
   X_test        : (50, 31)
